### Neighbourhood Characteristics and House Prices

This project explores how crime, deprivation and public transport accessibility are associated with median house prices across London neighbourhoods at LSOA level.

The analysis combines data from ONS (house prices), the Metropolitan Police, the English Indices of Deprivation and Transport for London.

In [2]:
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
import requests

### Data Loading

In [3]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def download_if_needed(url, path):
    """Download a file if it does not already exist."""
    if not path.exists():
        print(f"Downloading {path.name}...")
        response = requests.get(url)
        response.raise_for_status()
        path.write_bytes(response.content)


def extract_if_needed(zip_path, extract_dir):
    """Extract a ZIP file if the destination folder does not exist."""
    if not extract_dir.exists():
        print(f"Extracting {zip_path.name}...")
        with ZipFile(zip_path, "r") as zip_file:
            zip_file.extractall(extract_dir)

ptal_url = 'https://data.london.gov.uk/download/24rz6/77d9b319-931e-4090-bf8e-f578938bd352/LSOA2011%20AvPTAI2015.csv'
ptal_path =  DATA_DIR / "location.csv"

crime_url = 'https://data.london.gov.uk/download/exy3m/vm7/MPS%20LSOA%20Level%20Crime%20(Historical).csv'
crime_path = DATA_DIR / "crime.csv"

deprivation_url = "https://assets.publishing.service.gov.uk/media/5dc407b440f0b6379a7acc8d/File_7_-_All_IoD2019_Scores__Ranks__Deciles_and_Population_Denominators_3.csv"
deprivation_path = DATA_DIR / "deprivation.csv" 

house_url = "https://www.ons.gov.uk/file?uri=/peoplepopulationandcommunity/housing/datasets/medianpricepaidbylowerlayersuperoutputareahpssadataset46/current/hpssadataset46medianpricepaidforresidentialpropertiesbylsoa.zip"
house_prices_zip = DATA_DIR / "house_prices.zip"
house_prices_dir = DATA_DIR / "house_prices"

download_if_needed(ptal_url,ptal_path)
download_if_needed(crime_url,crime_path)
download_if_needed(deprivation_url,deprivation_path)
download_if_needed(house_url, house_prices_zip)
extract_if_needed(house_prices_zip, house_prices_dir)

In [4]:
house_prices_file = next(house_prices_dir.glob("*.xls"))

ptal_df = pd.read_csv(ptal_path)
crime_df = pd.read_csv(crime_path)
imd_df = pd.read_csv(deprivation_path)
prices_df = pd.read_excel(house_prices_file, sheet_name="1a", header=5) #sheet '1a' contains the median prices.

### Initial Exploratory Data Analysis and Data Preparation
Our first step in EDA is to understand the shape, data types, and missing values present in each dataset before we attempt to merge them.

##### **1.House Price Data**

The ONS dataset contains median residential property prices by LSOA, with values reported for rolling 12-month periods ending each quarter. We first inspect the dataset structure, data types and missing values. 

In [5]:
print("Shape:", prices_df.shape)
print('Columns:')
print(prices_df.columns)
print("dtypes:")
print(prices_df.dtypes)
print("Total nulls:", prices_df.isna().sum().sum())
print("Duplicated:", prices_df["LSOA code"].duplicated().sum())
prices_df.head()

Shape: (34753, 114)
Columns:
Index(['Local authority code', 'Local authority name', 'LSOA code',
       'LSOA name', 'Year ending Dec 1995', 'Year ending Mar 1996',
       'Year ending Jun 1996', 'Year ending Sep 1996', 'Year ending Dec 1996',
       'Year ending Mar 1997',
       ...
       'Year ending Dec 2020', 'Year ending Mar 2021', 'Year ending Jun 2021',
       'Year ending Sep 2021', 'Year ending Dec 2021', 'Year ending Mar 2022',
       'Year ending Jun 2022', 'Year ending Sep 2022', 'Year ending Dec 2022',
       'Year ending Mar 2023'],
      dtype='str', length=114)
dtypes:
Local authority code       str
Local authority name       str
LSOA code                  str
LSOA name                  str
Year ending Dec 1995    object
                         ...  
Year ending Mar 2022    object
Year ending Jun 2022    object
Year ending Sep 2022    object
Year ending Dec 2022    object
Year ending Mar 2023    object
Length: 114, dtype: object
Total nulls: 0
Duplicated: 0


,Local authority code,Local authority name,LSOA code,LSOA name,Year ending Dec 1995,Year ending Mar 1996,Year ending Jun 1996,Year ending Sep 1996,Year ending Dec 1996,Year ending Mar 1997,...,Year ending Dec 2020,Year ending Mar 2021,Year ending Jun 2021,Year ending Sep 2021,Year ending Dec 2021,Year ending Mar 2022,Year ending Jun 2022,Year ending Sep 2022,Year ending Dec 2022,Year ending Mar 2023
0,E06000001,Hartlepool,E01011949,Hartlepool 009A,34750,34500,30500,30000,29950,29000,...,88000,81500,80500,89000,101500,94500,113000,97500,102500,106500
1,E06000001,Hartlepool,E01011950,Hartlepool 008A,25000,25000,25300,25625,25000,24800,...,29750,33000,47000,49999,50159,50159,46000,43500,42000,43500
2,E06000001,Hartlepool,E01011951,Hartlepool 007A,27000,27000,27250,28950,28500,28950,...,50000,51500,53000,58573.5,60000,62999,61499.5,60000,65499.5,66000
3,E06000001,Hartlepool,E01011952,Hartlepool 002A,44500,44500,30000,26675,26000,25500,...,85000,:,83500,83000,80000,76000,75000,75000,70000,60000
4,E06000001,Hartlepool,E01011953,Hartlepool 002B,22000,27000,27000,20600,20000,19500,...,:,:,:,95000,92500,95000,95000,92500,93750,92500



The dataset contains 34,753 LSOA records and 114 columns. 
The geographical fields (`Local authority code`, `Local authority name`, `LSOA code`, and `LSOA name`) are correctly stored as text. Only London LSOAs are required for the analysis, so the dataset is filtered using London local authority codes beginning with `E09`. 

The quarterly house price columns are currently stored as `object` instead of numeric types. This is because the dataset uses `:` to represent missing values, causing pandas to interpret these columns as strings. We will replace `:` with missing values (`NaN`) and convert house price columns to numeric format.

In [6]:

prices_london = prices_df[prices_df["Local authority code"].str.startswith("E09")].copy()

# Identify price columns
price_cols = prices_london.columns[4:]

# Replace ONS missing value marker ":" with NaN and convert to numeric
prices_london[price_cols] = (prices_london[price_cols].replace(':', np.nan).apply(pd.to_numeric))

# Check dtypes
print(prices_london.dtypes)
print(prices_london.shape)


Local authority code        str
Local authority name        str
LSOA code                   str
LSOA name                   str
Year ending Dec 1995    float64
                         ...   
Year ending Mar 2022    float64
Year ending Jun 2022    float64
Year ending Sep 2022    float64
Year ending Dec 2022    float64
Year ending Mar 2023    float64
Length: 114, dtype: object
(4835, 114)


**Handling Missing House Price Values**

The analysis uses the median house price for the year ending December 2022. We first check how many December 2022 values are missing and, for those LSOAs, identify the most recent available house price from other reporting periods. This helps determine how much recent information is available for estimating missing December 2022 values and which observations do not contain sufficient data for reliable imputation.

In [7]:

missing_prices = prices_london["Year ending Dec 2022"].isna().sum()
print(f"Missing Dec 2022 prices: {missing_prices:,}, {missing_prices / len(prices_london):.2%}")

# Find the closest available observation for each missing row

remaining = prices_london[prices_london["Year ending Dec 2022"].isna()].copy()

check_cols = [
    "Year ending Mar 2023",
    "Year ending Sep 2022",
    "Year ending Jun 2022",
    "Year ending Mar 2022",
    "Year ending Dec 2021",
    "Year ending Sep 2021",
    "Year ending Jun 2021",
    "Year ending Mar 2021",
    "Year ending Dec 2020",
    "Year ending Sep 2020",
    "Year ending Jun 2020",
    "Year ending Mar 2020",
    "Year ending Dec 2019",
    "Year ending Sep 2019",
    "Year ending Jun 2019",
    "Year ending Mar 2019",
]

for col in check_cols:
    available = remaining[col].notna().sum()
    print(f"{col}: {available} available")
    # Remove rows where this value is available, so the next step checks only the rows still unresolved
    remaining = remaining[remaining[col].isna()]

print(f"Still missing after Mar 2019: {len(remaining)}")

Missing Dec 2022 prices: 339, 7.01%
Year ending Mar 2023: 52 available
Year ending Sep 2022: 45 available
Year ending Jun 2022: 36 available
Year ending Mar 2022: 72 available
Year ending Dec 2021: 26 available
Year ending Sep 2021: 22 available
Year ending Jun 2021: 8 available
Year ending Mar 2021: 2 available
Year ending Dec 2020: 2 available
Year ending Sep 2020: 5 available
Year ending Jun 2020: 5 available
Year ending Mar 2020: 4 available
Year ending Dec 2019: 2 available
Year ending Sep 2019: 3 available
Year ending Jun 2019: 4 available
Year ending Mar 2019: 3 available
Still missing after Mar 2019: 48


There are **339 missing December 2022 house-price values**. After checking the closest available observations, I decided to impute values where data is available from **September 2021 onwards** (**253 rows**) and leave the remaining **86 rows** as missing. September 2021 was chosen as the cutoff because data availability drops sharply for earlier periods.

Two imputation methods were used:
- Where both **September 2022** and **March 2023** values are available, December 2022 is estimated by linear interpolation using the midpoint between the two values.
- For the remaining rows, the closest available observation is adjusted using the **median house-price change within the same London borough** over the corresponding period. For example, if the latest available value is June 2022, it is adjusted using the median June-to-December 2022 price change for that borough.

Rows that still have no house-price value after imputation are retained for now so that geographic matching across datasets is not affected. They will be removed after the datasets are merged.

In [8]:
# Validate interpolation using rows where Sep 2022, Dec 2022 and Mar 2023 are known
interp_test = prices_london.dropna(subset=["Year ending Sep 2022", "Year ending Dec 2022", "Year ending Mar 2023"]).copy()
interp_test["predicted_price"] = (interp_test["Year ending Sep 2022"] + interp_test["Year ending Mar 2023"])/2
interp_test["error_pct"] =  abs(interp_test["predicted_price"] - interp_test["Year ending Dec 2022"])/interp_test["Year ending Dec 2022"]* 100

print("Interpolation median error:", round(interp_test["error_pct"].median(), 2),"%")

# Validate the imputation method by predicting known Dec 2022 prices
# from earlier observations and comparing predictions with actual values
periods = [
    "Year ending Mar 2023",
    "Year ending Sep 2022",
    "Year ending Jun 2022",
    "Year ending Mar 2022",
    "Year ending Dec 2021",
    "Year ending Sep 2021",
]

validation_results = []
borough_growth_rates = {}
for period in periods:
    test_df = prices_london.dropna(subset=[period, "Year ending Dec 2022"]).copy()

    # Calculate price growth for each LSOA
    test_df["growth_ratio"] = (test_df["Year ending Dec 2022"] / test_df[period])

    # Calculate and save median growth for each borough
    borough_growth_rates[period] = (test_df.groupby("Local authority name")["growth_ratio"].median())

    # Predict Dec 2022 price
    test_df["predicted_price"] = (test_df[period]* test_df["Local authority name"].map(borough_growth_rates[period]))

    # Calculate prediction error
    test_df["error_pct"] = (abs(test_df["predicted_price"]- test_df["Year ending Dec 2022"])/test_df["Year ending Dec 2022"]* 100)

    validation_results.append({
        "Closest available": period.replace("Year ending ", ""),
        "Rows tested": len(test_df),
        "Median error %": test_df["error_pct"].median(),
    })

validation_df = pd.DataFrame(validation_results)
validation_df.round(2)

Interpolation median error: 2.7 %


,Closest available,Rows tested,Median error %
0,Mar 2023,4397,3.04
1,Sep 2022,4397,3.45
2,Jun 2022,4318,5.78
3,Mar 2022,4388,7.62
4,Dec 2021,4401,8.71
5,Sep 2021,4412,8.67


The median prediction error increases as the last available observation becomes older, from 3.04% for March 2023 to around 8.7% for September-December 2021. Given that only a small proportion of house prices require imputation, a median error of around 8.7% was considered acceptable for this analysis.

In [9]:
# Create final house price column
prices_london["House Price 2022"] = prices_london["Year ending Dec 2022"]

# First, interpolate where values exist on both sides of Dec 2022
interpolation_mask = (
    prices_london["House Price 2022"].isna()
    & prices_london["Year ending Sep 2022"].notna()
    & prices_london["Year ending Mar 2023"].notna()
)

prices_london.loc[interpolation_mask, "House Price 2022"] = (
    prices_london.loc[interpolation_mask, "Year ending Sep 2022"]
    + prices_london.loc[interpolation_mask, "Year ending Mar 2023"]
) / 2

# Impute remaining values using borough-level median price changes
for period in periods:
    mask = (prices_london["House Price 2022"].isna() & prices_london[period].notna())

    prices_london.loc[mask, "House Price 2022"] = (
        prices_london.loc[mask, period] * prices_london.loc[mask, "Local authority name"]
        .map(borough_growth_rates[period])
    )

print("Still missing:", prices_london["House Price 2022"].isna().sum())

Still missing: 86


C:\Users\Olesya Drozhzhina\AppData\Local\Temp\ipykernel_6552\2337584392.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  prices_london["House Price 2022"] = prices_london["Year ending Dec 2022"]


In [10]:
# keep only necessary columns

prices_london = prices_london[
    [
        "Local authority code",
        "Local authority name",
        "LSOA code",
        "LSOA name",
        "House Price 2022",
    ]
]
print(prices_london.shape)
print(prices_london["House Price 2022"].isna().sum())
prices_london.head()


(4835, 5)
86


,Local authority code,Local authority name,LSOA code,LSOA name,House Price 2022
28009,E09000001,City of London,E01000001,City of London 001A,880000.0
28010,E09000001,City of London,E01000002,City of London 001B,850000.0
28011,E09000001,City of London,E01000003,City of London 001C,540000.0
28012,E09000001,City of London,E01000005,City of London 001E,NaN
28013,E09000001,City of London,E01032739,City of London 001F,756250.0


##### **2.Crime Data**

In [11]:
print("Shape:", crime_df.shape)
print("Missing values:", crime_df.isna().sum().sum())
print("Unique LSOAs:", crime_df["LSOA Code"].nunique())
print("Crime groups:", crime_df["Group"].nunique())
print("Crime subgroups:", crime_df["SubGroup"].nunique())

print(
    "Duplicate LSOA/group/subgroup combinations:",
    crime_df.duplicated(subset=["LSOA Code", "Group", "SubGroup"]).sum()
)
print("dtypes:")
print(crime_df.dtypes)
crime_df.head()

Shape: (120796, 53)
Missing values: 0
Unique LSOAs: 4991
Crime groups: 10
Crime subgroups: 31
Duplicate LSOA/group/subgroup combinations: 0
dtypes:
LSOA Code      str
LSOA Name      str
Borough        str
Group          str
SubGroup       str
202007       int64
202008       int64
202009       int64
202010       int64
202011       int64
202012       int64
202101       int64
202102       int64
202103       int64
202104       int64
202105       int64
202106       int64
202107       int64
202108       int64
202109       int64
202110       int64
202111       int64
202112       int64
202201       int64
202202       int64
202203       int64
202204       int64
202205       int64
202206       int64
202207       int64
202208       int64
202209       int64
202210       int64
202211       int64
202212       int64
202301       int64
202302       int64
202303       int64
202304       int64
202305       int64
202306       int64
202307       int64
202308       int64
202309       int64
202310       int

,LSOA Code,LSOA Name,Borough,Group,SubGroup,202007,202008,202009,202010,202011,...,202309,202310,202311,202312,202401,202402,202403,202404,202405,202406
0,E01000005,City of London 001E,E09000001,BURGLARY,BURGLARY BUSINESS AND COMMUNITY,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,E01000005,City of London 001E,E09000001,PUBLIC ORDER OFFENCES,PUBLIC FEAR ALARM OR DISTRESS,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,E01000005,City of London 001E,E09000001,THEFT,BICYCLE THEFT,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,E01000005,City of London 001E,E09000001,THEFT,OTHER THEFT,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
4,E01000005,City of London 001E,E09000001,THEFT,THEFT FROM THE PERSON,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1



The crime dataset contains monthly crime counts by LSOA, crime group and subgroup. Initial checks show no missing values, duplicate LSOA/group/subgroup combinations or incorrect data types.

For this analysis, only crime data from **2022** is required. The monthly 2022 values will be summed to create an annual crime count. Crime subgroups will then be combined into their main crime groups, and the data will be pivoted so that each LSOA has one row with separate columns for each crime group.

The dataset uses LSOA 2021 codes, which will later be converted to LSOA 2011 codes to match the other datasets.

In [12]:
# Select only the monthly crime columns for 2022
crime_2022_cols = [col for col in crime_df.columns if str(col).startswith("2022")]

# Sum monthly crime counts for 2022
crime_df["Crime Count 2022"] = crime_df[crime_2022_cols].sum(axis=1)

# Create a smaller dataframe keeping only relevant columns
crime_2022_df = crime_df[["LSOA Code", "LSOA Name", "Borough", "Group", "Crime Count 2022"]]

# Combine SubGroups into their main crime Group
crime_2022_df = (crime_2022_df.groupby(["LSOA Code", "LSOA Name", "Borough", "Group"],as_index=False)["Crime Count 2022"].sum())

# Pivot crime groups into separate columns
crime_2022_df = (crime_2022_df.pivot(
        index=["LSOA Code", "LSOA Name", "Borough"],
        columns="Group",
        values="Crime Count 2022"
    ).fillna(0).reset_index()
)

# Remove the column-axis name added by pivot()
crime_2022_df.columns.name = None

# Convert crime counts to integers
crime_cols = crime_2022_df.columns.difference(["LSOA Code", "LSOA Name", "Borough"])
crime_2022_df[crime_cols] = crime_2022_df[crime_cols].astype(int)

crime_2022_df.head()

,LSOA Code,LSOA Name,Borough,ARSON AND CRIMINAL DAMAGE,BURGLARY,DRUG OFFENCES,MISCELLANEOUS CRIMES AGAINST SOCIETY,POSSESSION OF WEAPONS,PUBLIC ORDER OFFENCES,ROBBERY,THEFT,VEHICLE OFFENCES,VIOLENCE AGAINST THE PERSON
0,E01000005,City of London 001E,E09000001,0,1,0,0,0,0,0,0,0,0
1,E01000006,Barking and Dagenham 016A,E09000002,6,3,17,5,2,4,10,12,26,34
2,E01000007,Barking and Dagenham 015A,E09000002,23,9,72,5,9,41,24,112,23,152
3,E01000008,Barking and Dagenham 015B,E09000002,14,14,27,3,5,16,5,27,32,67
4,E01000009,Barking and Dagenham 016B,E09000002,22,5,58,2,2,17,12,54,32,94


##### **3.Index of Multiple Deprivation (IMD) Data**

The IMD 2019 dataset contains deprivation measures for LSOAs across England, including the overall Index of Multiple Deprivation and seven deprivation domains. We first check the dataset structure, data types, missing values and duplicate LSOA codes. 

In [13]:
print("Shape:", imd_df.shape)
print("Missing values:", imd_df.isna().sum().sum())
print("Duplicate LSOAs:", imd_df["LSOA code (2011)"].duplicated().sum())
print(imd_df.dtypes)

imd_df.head()

Shape: (32844, 57)
Missing values: 0
Duplicate LSOAs: 0
LSOA code (2011)                                                                                          str
LSOA name (2011)                                                                                          str
Local Authority District code (2019)                                                                      str
Local Authority District name (2019)                                                                      str
Index of Multiple Deprivation (IMD) Score                                                             float64
Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)                                     int64
Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)                      int64
Income Score (rate)                                                                                   float64
Income Rank (where 1 is most deprived)                          

,LSOA code (2011),LSOA name (2011),Local Authority District code (2019),Local Authority District name (2019),Index of Multiple Deprivation (IMD) Score,Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived),Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs),Income Score (rate),Income Rank (where 1 is most deprived),Income Decile (where 1 is most deprived 10% of LSOAs),...,Indoors Sub-domain Rank (where 1 is most deprived),Indoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Outdoors Sub-domain Score,Outdoors Sub-domain Rank (where 1 is most deprived),Outdoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Total population: mid 2015 (excluding prisoners),Dependent Children aged 0-15: mid 2015 (excluding prisoners),Population aged 16-59: mid 2015 (excluding prisoners),Older population aged 60 and over: mid 2015 (excluding prisoners),Working age population 18-59/64: for use with Employment Deprivation Domain (excluding prisoners)
0,E01000001,City of London 001A,E09000001,City of London,6.208,29199,9,0.007,32831,10,...,16364,5,1.503,1615,1,1296,175,656,465,715
1,E01000002,City of London 001B,E09000001,City of London,5.143,30379,10,0.034,29901,10,...,22676,7,1.196,2969,1,1156,182,580,394,620
2,E01000003,City of London 001C,E09000001,City of London,19.402,14915,5,0.086,18510,6,...,17318,6,2.207,162,1,1350,146,759,445,804
3,E01000005,City of London 001E,E09000001,City of London,28.652,8678,3,0.211,6029,2,...,25218,8,1.769,849,1,1121,229,692,200,683
4,E01000006,Barking and Dagenham 016A,E09000002,Barking and Dagenham,19.837,14486,5,0.117,14023,5,...,14745,5,0.969,4368,2,2040,522,1297,221,1285


The dataset contains no missing values or duplicate LSOA codes, and the variables have appropriate data types, so no additional data cleaning is required.

The dataset contains a large number of ranks, deciles and sub-domain measures that are not needed for this analysis. We therefore retain only London LSOAs, the overall IMD score, the seven main deprivation domain scores and total population.

In [14]:
# Keep only London LSOAs
imd_london = imd_df[imd_df["Local Authority District code (2019)"].str.startswith("E09")].copy()

# Keep only variables required for the analysis
imd_keep = [
    "LSOA code (2011)",
    "LSOA name (2011)",
    "Local Authority District code (2019)",
    "Local Authority District name (2019)",
    
    "Index of Multiple Deprivation (IMD) Score",
    "Income Score (rate)",
    "Employment Score (rate)",
    "Education, Skills and Training Score",
    "Health Deprivation and Disability Score",
    "Crime Score",
    "Barriers to Housing and Services Score",
    "Living Environment Score",
    
    "Total population: mid 2015 (excluding prisoners)"
]

imd_london = imd_london[imd_keep]

imd_london.shape

(4835, 13)

##### **4.Public Transport Accessibility Levels (PTAL) Data**

The PTAL dataset contains public transport accessibility measures for London LSOAs. It includes the average Public Transport Accessibility Index, the corresponding PTAL category, and the upper and lower accessibility index values.

We first check the dataset structure, data types, missing values and duplicate LSOA codes. As the dataset already contains only London LSOAs, no geographic filtering is required.

In [15]:
print("Shape:", ptal_df.shape)
print("Missing values:", ptal_df.isna().sum().sum())
print("Duplicate LSOAs:", ptal_df["LSOA2011"].duplicated().sum())
print(ptal_df.dtypes)
ptal_df.head()

Shape: (4835, 5)
Missing values: 0
Duplicate LSOAs: 0
LSOA2011          str
AvPTAI2015    float64
PTAL              str
PTAIHigh      float64
PTAILow       float64
dtype: object


,LSOA2011,AvPTAI2015,PTAL,PTAIHigh,PTAILow
0,E01000001,69.8233,6b,97.4435,35.9190
1,E01000002,83.7820,6b,117.9120,66.3503
2,E01000003,41.7417,6b,49.5318,37.3635
3,E01000005,85.8893,6b,120.8470,45.9168
4,E01000006,22.4558,5,34.1054,0.0000


The dataset contains no missing values or duplicate LSOA codes, and the variables have appropriate data types, so no additional cleaning is required.

For the analysis, the continuous **AvPTAI2015** measure and the categorical **PTAL** classification will be retained. The upper and lower accessibility index values are not required and will be removed.

In [16]:
ptal_df = ptal_df[["LSOA2011","AvPTAI2015","PTAL"]].copy()

## Converting 2021 LSOA to 2011 LSOA

The house price, IMD and PTAL datasets use **LSOA 2011** codes, while the crime dataset uses **LSOA 2021** codes. To merge the datasets using a common geographic identifier, the crime data must first be converted to LSOA 2011 using the official **ONS LSOA 2011–2021 lookup table**.

Before applying the lookup, we check its mapping relationships to determine whether the conversion is one-to-one and whether all LSOA codes required for the analysis are present. This allows any boundary changes or missing mappings to be identified and resolved before the datasets are merged.


In [17]:
# download lookup table
look_up_url = 'https://open-geography-portalx-ons.hub.arcgis.com/api/download/v1/items/b684a0dbf786473f9563ec0616da2f8b/csv?layers=0'
look_up_path = DATA_DIR / "lookup.csv" 
download_if_needed(look_up_url,look_up_path)
lookup_df = pd.read_csv('data/lookup.csv')
lookup_df = lookup_df[['LSOA11CD','LSOA21CD']]

In [18]:
# check one-to-one match in lookup table
print("Lookup rows:", len(lookup_df))
print("Unique LSOA 2011:", lookup_df["LSOA11CD"].nunique())
print("Unique LSOA 2021:", lookup_df["LSOA21CD"].nunique())

print("LSOA 2021 mapping to multiple LSOA 2011:",
    (lookup_df.groupby("LSOA21CD")["LSOA11CD"].nunique() > 1).sum()
)
print("LSOA 2011 mapping to multiple LSOA 2021:",
    (lookup_df.groupby("LSOA11CD")["LSOA21CD"].nunique() > 1).sum()
)
# check that for codes from crime and house prices datasets there are matches in lookup table
missing_crime_codes = (set(crime_2022_df["LSOA Code"]) - set(lookup_df["LSOA21CD"]))
print("Crime LSOA 2021 codes missing from lookup:", len(missing_crime_codes))

missing_house_codes = (set(prices_london["LSOA code"]) - set(lookup_df["LSOA11CD"]))
print("House price LSOA2011 codes missing from lookup:", len(missing_house_codes))

Lookup rows: 34753
Unique LSOA 2011: 34753
Unique LSOA 2021: 34633
LSOA 2021 mapping to multiple LSOA 2011: 119
LSOA 2011 mapping to multiple LSOA 2021: 0
Crime LSOA 2021 codes missing from lookup: 181
House price LSOA2011 codes missing from lookup: 0


**Resolving LSOA Mapping Differences**

The lookup checks identified two issues that need to be resolved before converting the crime data.

First, the relationship between LSOA 2021 and LSOA 2011 is not completely one-to-one. Some LSOA2021 areas correspond to more than one LSOA2011 area because geographic boundaries changed between the two periods. For these cases, the crime count for the LSOA2021 will be divided equally between its corresponding LSOA2011 areas. For example, if one LSOA2021 maps to two LSOA2011 areas, each will receive 50% of its crime count.

Second, 181 LSOA2021 codes from the crime dataset are not present in the official lookup table. These missing mappings will be resolved using the **ONS Postcode Directory**, which contains both LSOA2011 and LSOA2021 codes for individual postcodes. For each missing LSOA2021, we identify which LSOA2011 contains the largest number of its postcodes and use this as its corresponding 2011 area.

The postcode-based mappings will then be added to the official lookup. Once the lookup covers all crime LSOA2021 codes, the mapping relationships will be checked again and allocation weights will be calculated. The completed lookup can then be applied to the crime dataset before aggregating the converted crime counts to one row per LSOA2011.

In [19]:
# Problem 1 - missing mapping for 181 LSOA 

ONSPD_URL = (
    "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services"
    "/ONS_Postcode_Directory_(February_2026)_for_the_UK_(Hosted_Table)"
    "/FeatureServer/0/query"
)

# Get the ObjectIds of postcode records belonging to the missing LSOA2021 areas
missing_codes = sorted(missing_crime_codes)
codes = "','".join(missing_codes)
where = f"lsoa21cd IN ('{codes}')"
params = {
    "where": where,
    "returnIdsOnly": "true",
    "f": "json"
}
response = requests.post(ONSPD_URL, data=params, timeout=60)
response.raise_for_status()
data = response.json()
object_ids = data["objectIds"]
print("Matching postcode records:", len(object_ids))

# Download the corresponding LSOA2021 and LSOA2011 codes in batches
postcode_records = []
id_batch_size = 100

for i in range(0, len(object_ids), id_batch_size):
    batch_ids = object_ids[i:i + id_batch_size]
    params = {
        "objectIds": ",".join(map(str, batch_ids)),
        "outFields": "lsoa21cd,lsoa11cd",
        "returnGeometry": "false",
        "f": "json"
    }
    response = requests.post(ONSPD_URL, data=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    for feature in data.get("features", []):
        row = feature["attributes"]
        postcode_records.append([row["lsoa21cd"], row["lsoa11cd"]])

postcode_df = pd.DataFrame(postcode_records,columns=["LSOA21CD", "LSOA11CD"])
print("Object IDs:", len(object_ids))
print("Postcode records downloaded:", len(postcode_records))
print("LSOA2021 codes found:",postcode_df["LSOA21CD"].nunique())
print("LSOA2021 codes still missing:", len(set(missing_codes) - set(postcode_df["LSOA21CD"])))
print("Postcode records without LSOA2011:", postcode_df["LSOA11CD"].isna().sum())

# Select the LSOA2011 containing the largest number of postcodes for each LSOA2021
postcode_counts = postcode_df.groupby(["LSOA21CD", "LSOA11CD"]).size().reset_index(name="postcode_count")
postcode_counts["total_postcodes"] = postcode_counts.groupby("LSOA21CD")["postcode_count"].transform("sum")
postcode_counts["postcode_share"] = postcode_counts["postcode_count"] / postcode_counts["total_postcodes"]
postcode_counts = postcode_counts.sort_values(["LSOA21CD", "postcode_count"],ascending=[True, False])
postcode_mapping = postcode_counts.drop_duplicates("LSOA21CD")
postcode_mapping.sort_values("postcode_share").head(20)


Matching postcode records: 11305
Object IDs: 11305
Postcode records downloaded: 11305
LSOA2021 codes found: 181
LSOA2021 codes still missing: 0
Postcode records without LSOA2011: 0


,LSOA21CD,LSOA11CD,postcode_count,total_postcodes,postcode_share
18,E01033911,E01000262,73,74,0.986486
0,E01033785,E01002293,53,53,1.000000
2,E01033787,E01002249,811,811,1.000000
1,E01033786,E01002249,16,16,1.000000
4,E01033792,E01000378,40,40,1.000000
5,E01033862,E01003109,78,78,1.000000
6,E01033865,E01033207,69,69,1.000000
3,E01033789,E01000432,41,41,1.000000
8,E01033868,E01003927,99,99,1.000000
9,E01033870,E01003978,34,34,1.000000


The postcode-based mappings were very clear. For each missing LSOA2021, the LSOA2011 containing the largest number of postcodes was selected. Even the weakest selected match assigned 73 of 74 postcodes (98.6%) to the same LSOA2011, indicating that the postcode-majority method provides an unambiguous mapping for these areas.

In [20]:
# Keep only the columns needed for the lookup
postcode_mapping_final = postcode_mapping[["LSOA21CD", "LSOA11CD"]].copy()

# Add recovered mappings to the official lookup
lookup_complete = pd.concat([lookup_df, postcode_mapping_final], ignore_index=True)

# Check that all crime LSOA2021 codes now have a mapping
still_missing = (set(crime_2022_df["LSOA Code"]) - set(lookup_complete["LSOA21CD"]))

print("Crime LSOA2021 codes still unmapped:", len(still_missing))
print("Duplicate mapping rows:", lookup_complete.duplicated(subset=["LSOA21CD", "LSOA11CD"]).sum())

# After adding the postcode-based mappings, check again 
# for many-to-one and one-to-many relationship in lookup table
lsoa21_to_11 = (lookup_complete.groupby("LSOA21CD")["LSOA11CD"].nunique())
lsoa11_to_21 = (lookup_complete.groupby("LSOA11CD")["LSOA21CD"].nunique())
print("LSOA2021 mapping to multiple LSOA2011:",(lsoa21_to_11 > 1).sum())
print("LSOA2011 receiving multiple LSOA2021:",(lsoa11_to_21 > 1).sum())

Crime LSOA2021 codes still unmapped: 0
Duplicate mapping rows: 0
LSOA2021 mapping to multiple LSOA2011: 119
LSOA2011 receiving multiple LSOA2021: 132


After adding the postcode-based mappings, all crime LSOA2021 codes have a corresponding LSOA2011 mapping and no duplicate mapping pairs are present.

The completed lookup contains **119 LSOA2021 areas that map to more than one LSOA2011 area**. These cases require the crime counts to be split between the corresponding 2011 areas.

It also contains **132 LSOA2011 areas that receive more than one LSOA2021 area**. These cases do not require splitting; the converted crime counts will be summed when the data is aggregated by LSOA2011.

In [21]:
# Problem 2 - one-to-many mapping 
# Create weights for LSOA2021 codes that map to multiple LSOA2011 codes

lookup_complete["n_lsoa11"] = (lookup_complete.groupby("LSOA21CD")["LSOA11CD"].transform("nunique"))
lookup_complete["weight"] = 1 / lookup_complete["n_lsoa11"]


crime_mapped = crime_2022_df.merge(
    lookup_complete,
    left_on="LSOA Code",
    right_on="LSOA21CD",
    how="left",
    validate="one_to_many"
)

crime_cols = crime_2022_df.columns.difference(["LSOA Code", "LSOA Name", "Borough"])
for col in crime_cols:
    crime_mapped[col] = (crime_mapped[col] * crime_mapped["weight"])


In [22]:
# Combine all crime records belonging to the same LSOA2011
crime_lsoa11 = crime_mapped.groupby("LSOA11CD", as_index=False)[crime_cols].sum()

crime_lsoa11["Total Crime 2022"] = crime_lsoa11[crime_cols].sum(axis=1)


### Checking LSOA Code Discrepancies
Next, we will check the distinct LSOA codes in each dataset to see if they match and identify any discrepancies before merging.

In [23]:
prices_lsoa = set(prices_london["LSOA code"])
crime_lsoa = set(crime_lsoa11["LSOA11CD"])
imd_lsoa = set(imd_london["LSOA code (2011)"])
ptal_lsoa = set(ptal_df["LSOA2011"])

common_lsoa = (prices_lsoa & crime_lsoa & imd_lsoa & ptal_lsoa)

print("House price LSOAs:", len(prices_lsoa))
print("Crime LSOAs:", len(crime_lsoa))
print("IMD LSOAs:", len(imd_lsoa))
print("PTAL LSOAs:", len(ptal_lsoa))
print("LSOAs present in all four:", len(common_lsoa))

print("\nNot in common set:")
print("House prices:", len(prices_lsoa - common_lsoa))
print("Crime:", len(crime_lsoa - common_lsoa))
print("IMD:", len(imd_lsoa - common_lsoa))
print("PTAL:", len(ptal_lsoa - common_lsoa))

House price LSOAs: 4835
Crime LSOAs: 4832
IMD LSOAs: 4835
PTAL LSOAs: 4835
LSOAs present in all four: 4832

Not in common set:
House prices: 3
Crime: 0
IMD: 3
PTAL: 3


In [24]:
print(ptal_lsoa - common_lsoa)

{'E01000001', 'E01000003', 'E01000002'}


After converting the crime data to LSOA 2011, the crime dataset contains three fewer LSOAs than the house price, IMD and PTAL datasets.

The missing areas are `E01000001`, `E01000002` and `E01000003`, all located in the **City of London**. These LSOAs are absent from the Metropolitan Police Service crime dataset because the City of London is policed separately by the **City of London Police**, rather than the Metropolitan Police Service.

As crime data is not available for these three LSOAs, they will be excluded from the final merged dataset.

## Merging the Four Datasets

The cleaned house price, crime, IMD and PTAL datasets will now be merged using the **LSOA 2011 code** as the common identifier. An inner merge will be used so that only LSOAs present in all four datasets are retained. Duplicate geographic identifiers will then be removed, column names will be standardised, and only variables required for the analysis will be kept.

In [25]:
merged_df = (crime_lsoa11
    .merge(prices_london,left_on="LSOA11CD",right_on="LSOA code",how="inner",validate="one_to_one")
    .merge(imd_london,left_on="LSOA11CD",right_on="LSOA code (2011)",how="inner",validate="one_to_one")
    .merge(ptal_df,left_on="LSOA11CD",right_on="LSOA2011",how="inner",validate="one_to_one")
)

# Remove rows where house price could not be estimated
merged_df = merged_df.dropna(subset=["House Price 2022"])

#Round the 2022 median house prices to the nearest whole number for consistency
merged_df["House Price 2022"] = merged_df["House Price 2022"].round().astype("Int64")

# Standardise column names
merged_df = merged_df.rename(columns={
    # Geography
    "Local authority code": "Local Authority Code",
    "Local authority name": "Local Authority Name",
    "LSOA name": "LSOA Name",

    # House prices
    "House Price 2022": "Median House Price 2022",

    # Crime
    "ARSON AND CRIMINAL DAMAGE": "Arson and Criminal Damage",
    "BURGLARY": "Burglary",
    "DRUG OFFENCES": "Drug Offences",
    "MISCELLANEOUS CRIMES AGAINST SOCIETY": "Miscellaneous Crimes Against Society",
    "POSSESSION OF WEAPONS": "Possession of Weapons",
    "PUBLIC ORDER OFFENCES": "Public Order Offences",
    "ROBBERY": "Robbery",
    "THEFT": "Theft",
    "VEHICLE OFFENCES": "Vehicle Offences",
    "VIOLENCE AGAINST THE PERSON": "Violence Against the Person",

    # Deprivation
    "Index of Multiple Deprivation (IMD) Score": "IMD Score",
    "Income Score (rate)": "Income Deprivation Score",
    "Employment Score (rate)": "Employment Deprivation Score",
    "Education, Skills and Training Score": "Education Deprivation Score",
    "Health Deprivation and Disability Score": "Health Deprivation Score",
    "Crime Score" : "Crime Deprivation Score", 
    "Barriers to Housing and Services Score": "Housing Deprivation Score",
    "Living Environment Score": "Living Environment Deprivation Score",
    "Total population: mid 2015 (excluding prisoners)": "Total Population 2015",

    # Transport
    "AvPTAI2015": "Average PTAI 2015"
})


# Order columns
final_cols = [
    "Local Authority Code",
    "Local Authority Name",
    "LSOA11CD",
    "LSOA Name",
    "Median House Price 2022",
    "Arson and Criminal Damage",
    "Burglary",
    "Drug Offences",
    "Miscellaneous Crimes Against Society",
    "Possession of Weapons",
    "Public Order Offences",
    "Robbery",
    "Theft",
    "Vehicle Offences",
    "Violence Against the Person",
    "Total Crime 2022",
    "IMD Score",
    "Income Deprivation Score",
    "Employment Deprivation Score",
    "Education Deprivation Score",
    "Health Deprivation Score",
    "Crime Deprivation Score",
    "Housing Deprivation Score",
    "Living Environment Deprivation Score",
    "Total Population 2015",
    "Average PTAI 2015",
    "PTAL"
]

final_df = merged_df[final_cols].copy()

print("Final shape:", final_df.shape)
print("Unique LSOAs:", final_df["LSOA11CD"].nunique())
print("Duplicate LSOAs:", final_df["LSOA11CD"].duplicated().sum())
print("Missing values:", final_df.isna().sum().sum())

Final shape: (4746, 27)
Unique LSOAs: 4746
Duplicate LSOAs: 0
Missing values: 0


After merging, only LSOAs present in all four datasets are retained. The remaining rows with missing house prices are then removed, as these values could not be estimated reliably during the house-price preparation stage.

The final dataset is checked for duplicate LSOAs and missing values before proceeding to the analysis.

## Saving the final dataframe as a csv file

In [26]:
output_path = DATA_DIR / "final_df.csv"
final_df.to_csv(output_path, index=False)

print(f"Saved to {output_path}")

Saved to data\final_df.csv
